# Notebook 9 – Building a Neural Network with PyTorch

We will build a small network with PyTorch, print its structure, and look at the weights and biases it needs to learn.

## 1.  `nn.Module`

`nn.Module` is the base class for a PyTorch model. We inherit from it so PyTorch can keep track of the model's layers and learnable values.

## 2. Constructor

The constructor is the `__init__` method. It runs when we create the model and is where we set up its layers.

## 3. Layers

Layers perform small calculations on input values. Connecting layers lets one layer pass its results to the next.

## 4. `nn.Linear`

`nn.Linear(input_size, output_size)` is a fully connected layer. For our data, `nn.Linear(2, 4)` accepts two input values—quantity and unit price—and produces four values, one for each hidden neuron.

## 5. Forward method

The `forward` method shows the order of calculations. When we call `model(example)`, PyTorch runs this method using the example data.

## 6. Model object

A model object is a working instance of our model class. We create it with `model = SimpleNeuralNetwork()` and pass input data to it to get a prediction.

## 7. Activation layer

An activation adds a non-linear step, helping the network learn more than straight-line relationships. ReLU changes negative values to zero and leaves positive values unchanged.

## 8. Output layer

The output layer gives the model's final answer. We want to predict one line total, so it has one output neuron. We leave ReLU off this final layer so it can return the number directly. At first, this prediction is only a starting guess because the model has not been trained.

## 9. Use a small example from our data

The `data.csv` file contains product purchase rows. We will use two columns as inputs: `Quantity` and `UnitPrice`. We will calculate `LineTotal = Quantity × UnitPrice` as the answer we want to predict.

For example, a row with quantity 6 and unit price 2.55 has a line total of 15.30. Below we load a few rows and choose one example. Our network will make an initial prediction, but it will not yet know the correct answer because we have not trained it.

The notebook and `data.csv` should be in the sprint folder when you run this example.

### 10. Input features

Each example has **2 input features**: quantity and unit price. Therefore, the first layer must accept 2 values.

### 11. Hidden neurons

For this small model, the first hidden layer has **4 neurons** and the second has **3 neurons**. These are simple example sizes, not rules that every network must follow.

### Output neuron

We are predicting one number, the product line total, so our output layer has **1 neuron**.

### 12. Architecture

**Quantity and UnitPrice → Hidden Layer (4) → ReLU → Hidden Layer (3) → ReLU → Output (1 line-total prediction)**. The output size of one layer must match the input size of the next layer.

In [1]:
import pandas as pd
from pathlib import Path

# The notebook and data.csv are in the same project folder.
data_path = Path("data.csv")
if not data_path.exists():
    data_path = Path(r"C:\Users\hemak\Sprint 9 – Deep Learning Fundamentals & Neural Networks\data.csv")

data = pd.read_csv(
    data_path,
    usecols=["Quantity", "UnitPrice"],
    nrows=5
).dropna()

# Calculate the example answer we want the model to learn.
data["LineTotal"] = data["Quantity"] * data["UnitPrice"]
print(data)

# Save the first real row for the model example below.
sample = data.iloc[0]
print()
print("Example answer (line total):", sample["LineTotal"])

   Quantity  UnitPrice  LineTotal
0         6       2.55      15.30
1         6       3.39      20.34
2         8       2.75      22.00
3         6       3.39      20.34
4         6       3.39      20.34

Example answer (line total): 15.299999999999999


In [2]:
import torch
import torch.nn as nn

class SimpleNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(2, 4)  # quantity and unit price -> 4 neurons
        self.hidden2 = nn.Linear(4, 3)  # 4 values -> 3 neurons
        self.output = nn.Linear(3, 1)   # 3 values -> one line-total prediction
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        x = self.output(x)
        return x

model = SimpleNeuralNetwork()

When one data row passes through the model, it follows this path: two input values → first linear layer → ReLU → second linear layer → ReLU → output layer. The final layer returns one number: the model's current line-total prediction. Since we have not trained it, do not expect that number to equal the actual line total yet.

## 13. Print and inspect the model architecture

Printing the model object shows the layers we created and their sizes. This is a quick way to check that the architecture matches our plan before training.

In [3]:
print(model)

SimpleNeuralNetwork(
  (hidden1): Linear(in_features=2, out_features=4, bias=True)
  (hidden2): Linear(in_features=4, out_features=3, bias=True)
  (output): Linear(in_features=3, out_features=1, bias=True)
  (relu): ReLU()
)


## Pass one data row through the network

The row is made into a 2D tensor with shape `1 × 2`: one example containing two features. The model returns a tensor with shape `1 × 1`: one predicted line total.

In [4]:
example = torch.tensor(
    [[float(sample["Quantity"]), float(sample["UnitPrice"])]],
    dtype=torch.float32
)
prediction = model(example)
actual_line_total = float(sample["LineTotal"])

print("Input features [quantity, unit price]:", example)
print("Input shape:", example.shape)
print("Model prediction (not trained yet):", prediction.item())
print("Actual line total from the data:", actual_line_total)
print("Output shape:", prediction.shape)

Input features [quantity, unit price]: tensor([[6.0000, 2.5500]])
Input shape: torch.Size([1, 2])
Model prediction (not trained yet): -0.8047353029251099
Actual line total from the data: 15.299999999999999
Output shape: torch.Size([1, 1])


The input values and correct answer come from the CSV row. The prediction will usually not match the actual line total yet: our model has random starting weights and has not been trained. This example is only showing how data goes into the network and one output comes back.

In [5]:
# A small example showing what one linear layer does to the real input.
# The example row has quantity and unit price as its two features.
toy_layer = nn.Linear(2, 2)
with torch.no_grad():
    toy_layer.weight.copy_(torch.tensor([[0.1, 1.0], [0.05, 0.5]]))
    toy_layer.bias.copy_(torch.tensor([0.0, 1.0]))

toy_output = toy_layer(example)
toy_activated_output = nn.ReLU()(toy_output)

print("Two hidden-neuron values before ReLU:", toy_output)
print("Values after ReLU:", toy_activated_output)

Two hidden-neuron values before ReLU: tensor([[3.1500, 2.5750]], grad_fn=<AddmmBackward0>)
Values after ReLU: tensor([[3.1500, 2.5750]], grad_fn=<ReluBackward0>)


This little layer has two input features and two neurons. Each neuron multiplies the input values by its weights, adds its bias, and produces one value. ReLU then changes any negative result to zero. These are only example weights; the real model has different starting weights that it can learn.

## 14. Model parameters

A **model parameter** is a value the network can learn, such as a weight or a bias. The network's parameters are stored inside its layers.

In [6]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 31
Trainable parameters: 31


## 15. Number of parameters

The first layer has `2 × 4 + 4 = 12` parameters (weights plus biases). The second has `4 × 3 + 3 = 15`, and the output layer has `3 × 1 + 1 = 4`. Together the model has **31 parameters**. ReLU layers have no learnable parameters.

## 16. Trainable parameters

A **trainable parameter** is a value PyTorch is allowed to adjust during training. In this model all 31 parameters are trainable. The code above counts these values for us.

## 17. Weight matrices

A **weight** controls how strongly an input affects a neuron. PyTorch stores a linear layer's weights as a matrix with shape `(output features, input features)`. For the first layer, the shape is `(4, 2)`: four neurons each receive the two input features, quantity and unit price.

## 18. Bias vectors

A **bias** is an extra learnable value added to a neuron's weighted sum. Each linear layer has one bias value per output neuron. The first layer therefore has four bias values.

In [7]:
for name, parameter in model.named_parameters():
    print(name, "shape:", tuple(parameter.shape))
print("\nFirst layer weight matrix:")
print(model.hidden1.weight)
print("First layer bias vector:")
print(model.hidden1.bias)

hidden1.weight shape: (4, 2)
hidden1.bias shape: (4,)
hidden2.weight shape: (3, 4)
hidden2.bias shape: (3,)
output.weight shape: (1, 3)
output.bias shape: (1,)

First layer weight matrix:
Parameter containing:
tensor([[ 0.0928,  0.6049],
        [ 0.4027,  0.2590],
        [-0.4917,  0.5320],
        [-0.4879, -0.4906]], requires_grad=True)
First layer bias vector:
Parameter containing:
tensor([-0.5958,  0.4573, -0.5139, -0.0914], requires_grad=True)


The first weight matrix has shape `(4, 2)`: four neurons each receive two input features, quantity and unit price. Its bias vector has four values, one per neuron. These values start with initial settings and can change during training.